# RNN Quest

Виконав: Роман, група ШІ-32 (roman.mydzhyn@example.com)

🗣️ **Пояснення:** Ставимо пакет курсу `lpnlp`.

In [1]:
!pip install --quiet --ignore-installed https://static.lp-nlp.com/pypi/lpnlp-2025.10.14-py3-none-any.whl requests==2.32.4

🗣️ **Пояснення:** Реєструємо початок квесту на сервері.

In [2]:
import lpnlp

lab = lpnlp.start(
    email="roman.mydzhyn@example.com",
    lab="quest_rnn"
)

Ваше завдання: https://static.lp-nlp.com/quest-rnn/d7071db0908e.zip. Удачі! █


1. Завантажуємо пакет із завданням за посиланням вище
2. Розпаковуємо архів
3. Читаємо README: у вагах Elman RNN закодовано повідомлення, його треба прочитати жадібним декодингом.

```
x_t = embedding[message[t - 1]]
h_t = tanh(W_h @ x_t + W_h_bias + U_h @ h_{t-1})
y_t = W_y @ h_t
message[t] = vocab_lookup(argmax(y_t))
```
Спецтокени: `[` початок, `]` кінець послідовності.

🗣️ **Пояснення:** Сервер дає посилання на архів із завданням. Завантажуємо його через `wget` і розпаковуємо.

In [3]:
package_url = lab.payload["package_url"]
print(package_url)
!wget -q {package_url} -O /content/quest.zip && unzip -o -q /content/quest.zip -d /content/quest
!ls -la /content/quest

https://static.lp-nlp.com/quest-rnn/d7071db0908e.zip
total 712
drwxr-xr-x 2 root root   4096 Nov 23 09:02 .
drwxr-xr-x 1 root root   4096 Nov 23 09:02 ..
-rw-rw-r-- 1 root root 130076 Oct 26  2025 embedding.weight.json
-rw-rw-r-- 1 root root    548 Oct 26  2025 quest.bin
-rw-rw-r-- 1 root root   3419 Oct 26  2025 README
-rw-rw-r-- 1 root root 215092 Oct 26  2025 U_h.weight.json
-rw-rw-r-- 1 root root    977 Oct 26  2025 vocab.json
-rw-rw-r-- 1 root root   1349 Oct 26  2025 W_h.bias.json
-rw-rw-r-- 1 root root 172110 Oct 26  2025 W_h.weight.json
-rw-rw-r-- 1 root root 178653 Oct 26  2025 W_y.weight.json


🗣️ **Пояснення:** Читаємо ваги натренованої Elman RNN із JSON: ембедінги `E`, матриці `W_h`, `U_h`, `W_y` і словник. Друкуємо розміри, щоб переконатися, що множення матриць зійдуться.

In [4]:
import json
import numpy as np

D = "/content/quest/"
def load(name):
    return np.array(json.load(open(D + name + ".json")))

vocab = json.load(open(D + "vocab.json"))
E   = load("embedding.weight")   # (vocab, emb)
W_h = load("W_h.weight")         # (hidden, emb)  -- формат torch.nn.Linear
b_h = load("W_h.bias")           # (hidden,)
U_h = load("U_h.weight")         # (hidden, hidden)
W_y = load("W_y.weight")         # (vocab, hidden)
print(E.shape, W_h.shape, b_h.shape, U_h.shape, W_y.shape, len(vocab))

(133, 128) (160, 128) (160,) (160, 160) (133, 160) 133


🗣️ **Пояснення:** Сам forward pass RNN. Стартуємо з токена `[` і нульового стану `h`. На кожному кроці беремо ембедінг попереднього символу, рахуємо новий стан `h = tanh(W_h·x + b + U_h·h)`, отримуємо logits `y = W_y·h` і жадібно (argmax) обираємо наступний символ. Зупиняємось на `]`. Так із ваг «читається» закодоване повідомлення.

In [5]:
def rnn_greedy_decode(max_len=1000):
    """Forward pass Elman RNN + жадібний декодинг."""
    h = np.zeros(U_h.shape[0])
    token = vocab.index("[")              # токен початку послідовності
    message = []
    for _ in range(max_len):
        x = E[token]                                  # x_t = embedding[message[t-1]]
        h = np.tanh(W_h @ x + b_h + U_h @ h)          # h_t
        y = W_y @ h                                   # y_t (логіти)
        token = int(np.argmax(y))                     # жадібний вибір
        if vocab[token] == "]":                       # кінець послідовності
            break
        message.append(vocab[token])
    return "".join(message)

message = rnn_greedy_decode()
print(message)

Назва виконавця: https://www.youtube.com/watch?v=dtSmUF0vMKE


Розкодоване повідомлення: `Назва виконавця: https://www.youtube.com/watch?v=dtSmUF0vMKE`.

За посиланням пісня **Gordosti** гурту **Ragapop** (канал RAGAPOP, "Gordosti · Ragapop"). Отже, відповідь: назва виконавця **Ragapop**.

4. Відправляємо відповідь

🗣️ **Пояснення:** Повідомлення вело на YouTube-кліп, виконавець Ragapop. Відправляємо відповідь на сервер.

In [6]:
lab.answer("Ragapop")

Відповідь правильна ✅
You did it! 🚀 Next step: https://tally.so/r/wMXM0p
